# Advanced Gradient Boosting: XGBoost, LightGBM & CatBoost

Gradient boosting is one of the most powerful ML techniques for tabular data. This notebook covers three industry-standard implementations.


In [ ]:
# Colab setup
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    for REPO in ['/content/drive/MyDrive/data-science-mastery', '/content/data-science-mastery']:
        if os.path.isdir(REPO):
            os.chdir(REPO)
            break
    print(f'Working dir: {os.getcwd()}')


## 1. Load Data


In [ ]:
df = pd.read_csv('Data/house_pricing.csv')
X = pd.get_dummies(df.drop('price', axis=1), columns=['location'], drop_first=True)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


## 2. XGBoost


In [ ]:
try:
    import xgboost as xgb
    print('XGBoost version:', xgb.__version__)
    
    start = time.time()
    xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    xgb_time = time.time() - start
    
    print(f'XGBoost RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_xgb)):,.2f}')
    print(f'XGBoost R2: {r2_score(y_test, y_pred_xgb):.4f}')
    print(f'Training time: {xgb_time:.2f}s')
except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')


## 3. LightGBM


In [ ]:
try:
    import lightgbm as lgb
    print('LightGBM version:', lgb.__version__)
    
    start = time.time()
    lgb_model = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)
    y_pred_lgb = lgb_model.predict(X_test)
    lgb_time = time.time() - start
    
    print(f'LightGBM RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_lgb)):,.2f}')
    print(f'LightGBM R2: {r2_score(y_test, y_pred_lgb):.4f}')
    print(f'Training time: {lgb_time:.2f}s')
except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')


## 4. CatBoost


In [ ]:
try:
    from catboost import CatBoostRegressor
    print('CatBoost loaded successfully!')
    
    start = time.time()
    cb_model = CatBoostRegressor(n_estimators=200, depth=6, learning_rate=0.1, random_seed=42, verbose=0)
    cb_model.fit(X_train, y_train)
    y_pred_cb = cb_model.predict(X_test)
    cb_time = time.time() - start
    
    print(f'CatBoost RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_cb)):,.2f}')
    print(f'CatBoost R2: {r2_score(y_test, y_pred_cb):.4f}')
    print(f'Training time: {cb_time:.2f}s')
except ImportError:
    print('CatBoost not installed. Run: pip install catboost')


## 5. Head-to-Head Comparison


In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'CatBoost'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        np.sqrt(mean_squared_error(y_test, y_pred_lgb)),
        np.sqrt(mean_squared_error(y_test, y_pred_cb))
    ],
    'R2': [
        r2_score(y_test, y_pred_xgb),
        r2_score(y_test, y_pred_lgb),
        r2_score(y_test, y_pred_cb)
    ],
})
print('\n=== Model Comparison ===')
print(results.round(4))


## When to Use Which?


In [ ]:
print('''
- XGBoost:  Gold standard. Works well everywhere. Best for smaller datasets.
- LightGBM: Fastest training. Best for large datasets (>10K rows).
            Uses histogram-based learning. Handles categoricals efficiently.
- CatBoost: Best with many categorical features. Symmetric trees,
            ordered boosting reduces overfitting. No encoding needed.
''')
